# AI Literacy: Start here

Orient yourself, set **widgets** (catalog / schema / model), then run preflight checks before Lab 1.

This notebook is for **workshop participants**. Set your widgets, confirm preflight PASS, then continue to Lab 1.

## What we build today

An **Auto-Loan Servicing Assistant** for a collections-style question: find retail auto loans in a state that are past due in a given range, with a balance over a threshold, then summarize the highest-balance account using **governed** loan data.

**Without AI:** same tables → fixed SQL or a dashboard filter → same numbers. No plain-English routing.

Run the **next three cells** (install → config/widgets → diagram) so the runtime flow appears here before the rest of the lab.

## Install lab packages

Install MLflow and the Databricks SDK before config and preflight. Safe to re-run.

In [ ]:
%pip install -U -q "mlflow[databricks]>=3.4" "databricks-sdk>=0.60" openai httpx
dbutils.library.restartPython()

### Load config and widgets

Run the next cell once. Use the **widget bar at the top** to set catalog, schema, LLM endpoint, warehouse, filter state, and PII group. Change anytime, then re-run this config cell (and the diagram cell if you want the image again).

Docs: [Notebook widgets](https://learn.microsoft.com/en-us/azure/databricks/notebooks/widgets)

In [ ]:
%run ./_config

### Runtime flow (visual)

Same picture as the workshop README. You ask in English → the agent chooses lookups → lookups run on governed tables → you get a short grounded summary.

In [ ]:
show_diagram("what_we_build.png", "What we build: analyst → agent → UC tools → governed tables → summary")

## Why an agent for this problem

### What AI solves here

1. Plain English to the right lookups
2. Multi-step work in one ask
3. Short narrative from facts

### What AI does not solve here

1. Not a replacement for governed data
2. Not a free invent-the-answer box (Lab 3 catches trust of a wrong note)
3. Not required for a fixed daily report

## Words we will use

- **Governed data:** tables with permissions so not everyone sees everything
- **Tool / lookup:** a named, read-only lookup the agent may call (Unity Catalog functions)
- **Agent:** a language model plus written rules plus tools
- **Instructions:** the written rules the agent must follow
- **Evaluation:** scoring answers against what we expect
- **System of record:** structured fields we trust for facts, not free-text notes

## Lab path from here

1. **Lab 1** — Your schema, governed tables, three UC tools  
2. **Lab 2** — Playground + notebook agent (`run_agent`)  
3. **Lab 3** — Measure, catch the trap, fix instructions  
4. **Lab 4** (optional) — Same tables via Genie + ideation canvas  
5. **Lab 5** — What `_agent_core` is, and how this becomes a real agent  

Model widget: paste a `system.ai…` FQN **or** a legacy endpoint name. If preflight says Gateway unavailable, use the legacy name your admin gave you.


## Your resolved lab values

Confirm these match the widgets. You will paste some into AI Playground in Lab 2.

In [ ]:
print(f"""
  You are ............ {USER}
  Workshop catalog ... {CATALOG}
  Your schema ........ {FQ_PLAIN}
  LLM endpoint ....... {LLM_ENDPOINT}
  SQL warehouse ...... {WAREHOUSE_NAME}
  Hidden state ....... {HIDDEN_STATE}
  PII group .......... {PII_GROUP}
  MLflow experiment .. {EXPERIMENT_PATH}
""")

## Pre-flight checks

Each check prints PASS, FAIL, or WARN. **FAIL** blocks Lab 1.

These must already be true (your admin set them up before class):

- You can use the catalog widget and create a schema
- You can query the LLM endpoint and use the SQL warehouse
- Optional: PII viewer group exists for the presenter demo (you are usually **not** a member)

### Check helpers

Small helpers used by the checks below.

In [ ]:
from databricks.sdk import WorkspaceClient
import mlflow

w = WorkspaceClient()
results = []  # (name, status, detail, required)


def check(name, fn, required=True):
    """Run one probe; store PASS/FAIL/WARN without stopping the loop."""
    try:
        detail = fn()
        results.append((name, "PASS", detail or "", required))
    except Exception as e:
        results.append((name, "FAIL" if required else "WARN", str(e)[:220], required))

### Catalog and schema probes

In [ ]:
def catalog_access():
    try:
        n = len(spark.sql(f"SHOW SCHEMAS IN `{CATALOG}`").collect())
        return f"{n} schemas visible in `{CATALOG}`"
    except Exception as e:
        raise RuntimeError(
            f"Cannot USE CATALOG `{CATALOG}`. Ask the instructor to grant USE CATALOG "
            f"(and CREATE SCHEMA). ({e})"
        ) from e


def create_schema():
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ} COMMENT 'AI Literacy lab schema for {USER}'")
        spark.sql(f"COMMENT ON SCHEMA {FQ} IS 'AI Literacy lab schema for {USER}'")
        return f"{FQ_PLAIN} ready"
    except Exception as e:
        raise RuntimeError(
            f"CREATE SCHEMA denied on `{CATALOG}`. Ask for GRANT CREATE SCHEMA. ({e})"
        ) from e


def create_function():
    # Lab 1 needs CREATE FUNCTION for tools, row filter, and mask
    spark.sql(f"CREATE OR REPLACE FUNCTION {FQ}._lab_preflight() RETURNS INT RETURN 1")
    spark.sql(f"DROP FUNCTION IF EXISTS {FQ}._lab_preflight")
    return "CREATE FUNCTION works in your schema"

check("Catalog access (USE CATALOG)", catalog_access)
check("Can create your schema (CREATE SCHEMA)", create_schema)
check("Can create functions in your schema", create_function)

### Endpoint, warehouse, and MLflow probes

In [ ]:
def llm_endpoint():
    ok, detail, mode = probe_llm_access()
    if not ok:
        raise RuntimeError(detail)
    return detail


def sql_warehouse():
    x = resolve_sql_warehouse()
    return f"{x.name} id={x.id}"


def mlflow_home():
    mlflow.set_experiment(EXPERIMENT_PATH)
    return EXPERIMENT_PATH


check("LLM model reachable (UC Gateway or legacy endpoint)", llm_endpoint)
check("SQL warehouse visible (CAN USE)", sql_warehouse)
check("MLflow experiment in your home folder", mlflow_home)


### PII group probe (WARN only)

Attendees should usually **not** be members. The notebook never creates the group.

In [ ]:
def pii_group():
    member = bool(spark.sql(f"SELECT is_account_group_member('{PII_GROUP}')").first()[0])
    role = (
        "you ARE a member: you will see real names and hidden-state rows (presenter view)"
        if member else
        "you are NOT a member: names masked, hidden state filtered (expected attendee view)"
    )
    try:
        found = any(True for _ in w.groups.list(filter=f'displayName eq "{PII_GROUP}"'))
    except Exception as e:
        raise RuntimeError(
            f"Could not list account groups ({type(e).__name__}). Instructor must create "
            f"'{PII_GROUP}' and add only the presenter. {role}"
        ) from e
    if not found:
        raise RuntimeError(
            f"Account group '{PII_GROUP}' was not found. Admin creates it (presenter only). {role}"
        )
    return f"group '{PII_GROUP}' exists. {role}"

check(f"Account group '{PII_GROUP}' (instructor creates)", pii_group, required=False)

### Preflight summary

In [ ]:
for name, status, detail, _ in results:
    print(f"{status:5}  {name}\n       {detail}")

hard_fail = [n for n, s, _, req in results if req and s != "PASS"]
if hard_fail:
    raise AssertionError("Fix the FAIL items before Lab 1: " + "; ".join(hard_fail))

print("\nRequired checks passed. Open 01_setup_governed_data.")
print("If the group check is WARN, that is usually OK for participants.")

## Next

Open **`01_setup_governed_data`** and Run All. After Labs 1–3 (and optional 4), open **`05_from_lab_to_real_agent`**.
